# Homework: SARSA and Q-learning on FrozenLake

This notebook reinforces the material from the TD-control notes.
You need to implement the SARSA and Q-learning algorithms, compare their behavior, and investigate the effect of environment modifications and policy type.

## Learning objectives
- Implement the SARSA and Q-learning algorithms for finding the optimal policy
- Compare on-policy (SARSA) and off-policy (Q-learning) approaches
- Investigate the effect of a reward modification (penalty for falling into a hole)
- Study the difference between ε-greedy and softmax policies
- Formulate conclusions about the applicability of each method

## Theoretical background

### SARSA (on-policy)
The Q-function update uses the action that is actually taken by the policy:
$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha [r_{t+1} + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t)]$$

where $a_{t+1}$ is chosen from the current policy (e.g., ε-greedy).

### Q-learning (off-policy)
The Q-function update uses the maximizing action regardless of the behavior policy:
$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha [r_{t+1} + \gamma \max_a Q(s_{t+1}, a) - Q(s_t, a_t)]$$

### Key differences
- **SARSA**: accounts for exploration risk (action $a_{t+1}$ can be random due to ε)
- **Q-learning**: estimates the optimal policy directly, ignoring exploration
- **In practice**: SARSA is more conservative, Q-learning is more aggressive

## How to do the assignment
- Go from top to bottom; fill in your own code for each block marked `TODO`
- If you're running in Colab, install the dependencies
- Fix the random seeds for reproducibility
- At the end, fill in the section with questions and conclusions

### Environment setup

In [ ]:
# If you're working in Colab, uncomment the lines below
# !pip install gymnasium numpy matplotlib tqdm -q

In [ ]:
import random
from dataclasses import dataclass
from typing import Tuple, Callable

import gymnasium as gym
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

In [ ]:
SEED = 2024
WINDOW = 100  # Window size for averaging metrics

random.seed(SEED)
np.random.seed(SEED)

## 1. Modifying the FrozenLake environment

Let's create a wrapper for FrozenLake that adds a negative reward for falling into a hole.
This makes the task more realistic and allows us to explore the difference between conservative (SARSA) and aggressive (Q-learning) behavior.

**Task:** implement `ModifiedFrozenLakeEnv` — a wrapper that:
- Returns `hole_penalty` (e.g., -1.0) when falling into a hole
- Keeps the standard reward of +1 for reaching the goal
- Returns 0 for regular steps

In [ ]:
class ModifiedFrozenLakeEnv(gym.Wrapper):
    """Wrapper for FrozenLake with a negative reward for falling into a hole.

    FrozenLake 4x4 map:
        S F F F      S = Start
        F H F H      F = Frozen (ice, safe)
        F F F H      H = Hole (fall)
        H F F G      G = Goal (+1 reward)

    States are numbered 0-15, left to right, top to bottom.
    Actions: 0=Left, 1=Down, 2=Right, 3=Up
    """

    def __init__(self, env: gym.Env, hole_penalty: float = -1.0):
        """Initializes the wrapper with a penalty for holes.

        TODO: fill in the blanks (replace ... with the correct code)
        """
        # Step 1: call the parent class constructor
        super().__init__(env)

        # Step 2: store the penalty for falling into a hole
        self.hole_penalty = ...  # TODO: store hole_penalty

        # Step 3: get the environment map to determine the cell type
        # env.unwrapped.desc is a numpy array with characters b'S', b'F', b'H', b'G'
        self.desc = ...  # TODO: get the map description

    def step(self, action: int) -> Tuple[int, float, bool, bool, dict]:
        """Performs a step and modifies the reward when falling into a hole.

        TODO: fill in the blanks
        """
        # Step 1: perform the action in the original environment
        next_state, reward, terminated, truncated, info = self.env.step(action)

        # Step 2: determine the cell coordinates (row, col) from the state number
        # Formula: state = row * 4 + col, so row = state // 4, col = state % 4
        row = next_state // 4
        col = ...  # TODO: compute the column number

        # Step 3: check whether the cell is a hole
        # Hole character in the map: b'H'
        is_hole = (self.desc[row, col] == b'H')

        # Step 4: if the episode ended in a hole, apply the penalty
        if terminated and is_hole:
            reward = ...  # TODO: replace the reward with the penalty

        return next_state, float(reward), terminated, truncated, info


def make_env(seed: int = SEED, is_slippery: bool = False, hole_penalty: float = 0.0) -> gym.Env:
    """Creates FrozenLake with an optional reward modification.

    Args:
        seed: seed for reproducibility
        is_slippery: True = stochastic environment (slippery ice)
                     False = deterministic environment (easier to learn)
        hole_penalty: penalty for falling into a hole (0.0 = standard environment)
    """
    env = gym.make("FrozenLake-v1", is_slippery=is_slippery)
    if hole_penalty != 0.0:
        env = ModifiedFrozenLakeEnv(env, hole_penalty=hole_penalty)
    env.reset(seed=seed)
    return env

In [ ]:
# Test of the modified environment
# Run several episodes and check that the hole penalty works

test_env = make_env(hole_penalty=-1.0, is_slippery=False)
penalty_count = 0
total_episodes = 20

for episode in range(total_episodes):
    state, _ = test_env.reset(seed=SEED + 100 + episode)
    done = False

    while not done:
        # Random action
        action = test_env.action_space.sample()
        state, reward, terminated, truncated, _ = test_env.step(action)
        done = terminated or truncated

        # Check whether we got a penalty
        if terminated and reward < 0:
            penalty_count += 1
            break

test_env.close()

print(f"Episodes with penalty: {penalty_count} out of {total_episodes}")
print(f"If penalty_count > 0, then ModifiedFrozenLakeEnv works correctly!")

## 2. Policies: ε-greedy and softmax

Let's implement two exploration strategies:
1. **ε-greedy**: with probability ε choose a random action, otherwise the greedy one
2. **Softmax (Boltzmann)**: probabilities are proportional to $e^{Q(s,a)/\tau}$, where τ is the temperature

In [ ]:
def epsilon_greedy_action(Q: np.ndarray, state: int, epsilon: float) -> int:
    """Chooses an action according to the ε-greedy policy.

    Logic:
    - With probability ε: a random action (exploration)
    - With probability (1-ε): the best action from the Q-table (exploitation)

    TODO: fill in the blanks
    """
    if np.random.random() < epsilon:
        # Exploration: a random action from [0, n_actions)
        n_actions = Q.shape[1]  # Number of actions = number of columns in Q
        return ...  # TODO: return a random integer from 0 to n_actions-1
    else:
        # Exploitation: choose the action with the maximum Q-value
        return int(np.argmax(Q[state]))


def softmax_action(Q: np.ndarray, state: int, temperature: float = 1.0) -> int:
    """Chooses an action according to the softmax (Boltzmann) policy.

    Probability formula: P(a|s) = exp(Q(s,a)/τ) / Σ exp(Q(s,a')/τ)

    - Low temperature (τ→0): nearly greedy choice
    - High temperature (τ→∞): nearly uniform choice

    TODO: fill in the blanks
    """
    # Guard against division by zero
    temp = max(temperature, 1e-6)

    # Step 1: compute logits = Q-values / temperature
    logits = Q[state] / temp

    # Step 2: normalize for numerical stability (subtract the max)
    # This prevents overflow when computing exp() of large numbers
    logits = logits - np.max(logits)

    # Step 3: compute probabilities via softmax
    exp_values = np.exp(logits)
    probs = ...  # TODO: normalize exp_values (divide by the sum)

    # Step 4: sample an action according to the probabilities
    return int(np.random.choice(len(probs), p=probs))

## 3. SARSA implementation

SARSA is an on-policy algorithm that updates the Q-function based on the actions actually taken:
1. Choose $a_t$ from the current policy (e.g., ε-greedy)
2. Take $a_t$, get $(s_{t+1}, r_{t+1})$
3. Choose $a_{t+1}$ from the same policy
4. Update: $Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha [r_{t+1} + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t)]$

**Important:** action $a_{t+1}$ is chosen before the Q update, which makes the algorithm on-policy.

In [ ]:
@dataclass
class SARSAConfig:
    """Configuration for the SARSA algorithm."""
    gamma: float = 0.99           # Discount factor (how much we value future rewards)
    alpha: float = 0.1            # Learning rate
    epsilon_start: float = 1.0    # Initial ε value (lots of exploration)
    epsilon_end: float = 0.01     # Final ε value (little exploration)
    epsilon_decay: float = 0.999  # ε decay multiplier after each episode
    num_episodes: int = 10000     # Number of training episodes
    max_steps: int = 100          # Maximum steps per episode


def sarsa(env: gym.Env, config: SARSAConfig, use_softmax: bool = False, temperature: float = 1.0):
    """The SARSA (State-Action-Reward-State-Action) algorithm.

    ON-POLICY algorithm: we update Q using the action that we ACTUALLY take.

    Update formula:
        Q(s,a) ← Q(s,a) + α * [r + γ*Q(s',a') - Q(s,a)]
                              ↑ TD target    ↑ current estimate

        Where a' is the next action, chosen by the SAME policy!

    TODO: fill in the blanks

    Returns:
        Q: the trained Q-table of shape (n_states, n_actions)
        rewards_history: the average reward for every 100 episodes
        success_rate_history: the fraction of successful episodes for every 100 episodes
    """
    n_states = env.observation_space.n   # 16 states for FrozenLake 4x4
    n_actions = env.action_space.n       # 4 actions: Left, Down, Right, Up

    # Initialize the Q-table with zeros (or optimistically, with ones)
    Q = np.zeros((n_states, n_actions))

    # Metric history for plots
    rewards_history = []
    success_rate_history = []

    # Buffer for the moving average
    window_rewards = []
    window_success = []

    # Initial epsilon value
    epsilon = config.epsilon_start

    for episode in tqdm(range(config.num_episodes), desc="SARSA"):
        # === START OF EPISODE ===
        state, _ = env.reset()
        total_reward = 0.0
        is_success = False

        # SARSA: choose the FIRST action BEFORE the loop starts
        if use_softmax:
            action = softmax_action(Q, state, temperature)
        else:
            action = epsilon_greedy_action(Q, state, epsilon)

        for step in range(config.max_steps):
            # === ENVIRONMENT INTERACTION STEP ===

            # 1. Take the action, get the new state and reward
            next_state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward

            # 2. Choose the NEXT action (on-policy: same policy!)
            if use_softmax:
                next_action = softmax_action(Q, next_state, temperature)
            else:
                next_action = ...  # TODO: choose an action via ε-greedy

            # === Q-FUNCTION UPDATE ===

            # 3. Compute the TD target
            if terminated:
                # Terminal state: no future rewards
                td_target = reward
                if reward > 0:
                    is_success = True
            else:
                # SARSA formula: r + γ * Q(s', a')
                td_target = reward + config.gamma * Q[next_state, next_action]

            # 4. Compute the TD error
            td_error = td_target - Q[state, action]

            # 5. Update the Q-value
            # Formula: Q(s,a) ← Q(s,a) + α * TD_error
            Q[state, action] += ...  # TODO: apply the update formula

            # === TRANSITION TO THE NEXT STEP ===
            state = next_state
            action = next_action  # In SARSA the action is already chosen!

            if terminated or truncated:
                break

        # === END OF EPISODE ===

        # Decay epsilon (reduce exploration over time)
        epsilon = max(config.epsilon_end, epsilon * config.epsilon_decay)

        # Save episode metrics
        window_rewards.append(total_reward)
        window_success.append(1.0 if is_success else 0.0)

        # Every 100 episodes, save the average
        if (episode + 1) % WINDOW == 0:
            rewards_history.append(np.mean(window_rewards))
            success_rate_history.append(np.mean(window_success))
            window_rewards = []
            window_success = []

    return Q, rewards_history, success_rate_history

## 4. Q-learning implementation

Q-learning is an off-policy algorithm that directly estimates the optimal Q-function:
1. Choose $a_t$ from the behavior policy (e.g., ε-greedy)
2. Take $a_t$, get $(s_{t+1}, r_{t+1})$
3. Update: $Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha [r_{t+1} + \gamma \max_a Q(s_{t+1}, a) - Q(s_t, a_t)]$

**Key difference:** we use $\max_a Q(s_{t+1}, a)$ instead of $Q(s_{t+1}, a_{t+1})$.

In [ ]:
@dataclass
class QLearningConfig:
    """Configuration for the Q-learning algorithm."""
    gamma: float = 0.99           # Discount factor
    alpha: float = 0.1            # Learning rate
    epsilon_start: float = 1.0    # Initial ε
    epsilon_end: float = 0.01     # Final ε
    epsilon_decay: float = 0.999  # ε decay
    num_episodes: int = 10000
    max_steps: int = 100


def q_learning(env: gym.Env, config: QLearningConfig, use_softmax: bool = False, temperature: float = 1.0):
    """The Q-learning algorithm.

    OFF-POLICY algorithm: we update Q using the BEST possible action,
    regardless of which action we actually take.

    Update formula:
        Q(s,a) ← Q(s,a) + α * [r + γ*max_a' Q(s',a') - Q(s,a)]
                              ↑ TD target (optimistic)

        We use max instead of Q(s',a') — this is the key difference from SARSA!

    TODO: fill in the blanks

    Returns:
        Q: the trained Q-table
        rewards_history: history of average rewards
        success_rate_history: history of the success rate
    """
    n_states = env.observation_space.n
    n_actions = env.action_space.n

    Q = np.zeros((n_states, n_actions))

    rewards_history = []
    success_rate_history = []
    window_rewards = []
    window_success = []

    epsilon = config.epsilon_start

    for episode in tqdm(range(config.num_episodes), desc="Q-learning"):
        # === START OF EPISODE ===
        state, _ = env.reset()
        total_reward = 0.0
        is_success = False

        # Q-learning: we do NOT choose an action in advance (unlike SARSA)

        for step in range(config.max_steps):
            # === ENVIRONMENT INTERACTION STEP ===

            # 1. Choose an action (behavior policy)
            if use_softmax:
                action = softmax_action(Q, state, temperature)
            else:
                action = epsilon_greedy_action(Q, state, epsilon)

            # 2. Take the action
            next_state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward

            # === Q-FUNCTION UPDATE ===

            # 3. Compute the TD target (OFF-POLICY: we use max!)
            if terminated:
                td_target = reward
                if reward > 0:
                    is_success = True
            else:
                # Q-learning formula: r + γ * max_a' Q(s', a')
                # np.max(Q[next_state]) returns the maximum Q-value
                td_target = reward + config.gamma * ...  # TODO: max Q(s', a')

            # 4. Update the Q-value
            Q[state, action] += config.alpha * (td_target - Q[state, action])

            # === TRANSITION TO THE NEXT STEP ===
            state = next_state
            # In Q-learning we do NOT store next_action — we choose it on the next step

            if terminated or truncated:
                break

        # === END OF EPISODE ===
        epsilon = max(config.epsilon_end, epsilon * config.epsilon_decay)

        window_rewards.append(total_reward)
        window_success.append(1.0 if is_success else 0.0)

        if (episode + 1) % WINDOW == 0:
            rewards_history.append(np.mean(window_rewards))
            success_rate_history.append(np.mean(window_success))
            window_rewards = []
            window_success = []

    return Q, rewards_history, success_rate_history

## 5. Experiment A: SARSA vs Q-learning on standard FrozenLake

Let's compare both algorithms on the standard environment (without a reward modification).

### Goal of the experiment
Understand the difference between **on-policy** (SARSA) and **off-policy** (Q-learning) learning.

### Experiment plan
1. Create a `FrozenLake` environment without a hole penalty (`hole_penalty=0.0`)
2. Train SARSA with the `SARSAConfig()` configuration
3. Train Q-learning with the `QLearningConfig()` configuration
4. Compare:
   - Final success rate (% of successful episodes)
   - Convergence speed (learning curve)
   - Learned policies (4x4 action map)

### Expected results
- Q-learning should converge **faster** (optimistic updates via max)
- SARSA may be more **stable** (accounts for actual behavior)
- The final policies may **differ** near dangerous cells

In [ ]:
# ============================================================================
# EXPERIMENT A: SARSA vs Q-learning on standard FrozenLake
# ============================================================================
#
# TODO: fill in the blanks (...) and run the cell
# ============================================================================

# Step 1: Reset the seeds for reproducibility of results
np.random.seed(SEED)
random.seed(SEED)

# Step 2: Create algorithm configurations (use default values)
sarsa_config = SARSAConfig()
ql_config = QLearningConfig()

print("Training parameters:")
print(f"  Episodes: {sarsa_config.num_episodes}")
print(f"  γ (gamma): {sarsa_config.gamma}")
print(f"  α (alpha): {sarsa_config.alpha}")
print(f"  ε: {sarsa_config.epsilon_start} → {sarsa_config.epsilon_end}")
print()

# ============================================================================
# Step 3: Train SARSA
# ============================================================================
print("=" * 50)
print("Training SARSA...")
print("=" * 50)

# Create the standard environment (no hole penalty)
env_std = make_env(hole_penalty=0.0, is_slippery=False)

# Run SARSA training
# TODO: call the sarsa() function with the correct arguments
Q_sarsa_std, rewards_sarsa_std, success_sarsa_std = ...  # sarsa(env_std, sarsa_config)

env_std.close()

# Print the result
print(f"\nSARSA — final success rate: {success_sarsa_std[-1] * 100:.1f}%")

# ============================================================================
# Step 4: Train Q-learning
# ============================================================================
print("\n" + "=" * 50)
print("Training Q-learning...")
print("=" * 50)

# Reset the seeds for a fair comparison
np.random.seed(SEED)
random.seed(SEED)

# Create the same environment
env_std = make_env(hole_penalty=0.0, is_slippery=False)

# Run Q-learning training
# TODO: call the q_learning() function with the correct arguments
Q_ql_std, rewards_ql_std, success_ql_std = ...  # q_learning(env_std, ql_config)

env_std.close()

# Print the result
print(f"\nQ-learning — final success rate: {success_ql_std[-1] * 100:.1f}%")

# ============================================================================
# Step 5: Compare the policies
# ============================================================================
print("\n" + "=" * 50)
print("Comparing the learned policies")
print("=" * 50)

# Actions: 0=Left(←), 1=Down(↓), 2=Right(→), 3=Up(↑)
action_symbols = ['←', '↓', '→', '↑']

print("\nSARSA policy (optimal action for each state):")
sarsa_policy = np.argmax(Q_sarsa_std, axis=1).reshape(4, 4)
print(sarsa_policy)

print("\nQ-learning policy:")
ql_policy = np.argmax(Q_ql_std, axis=1).reshape(4, 4)
print(ql_policy)

# Check whether the policies match
if np.array_equal(sarsa_policy, ql_policy):
    print("\n✓ Policies are IDENTICAL")
else:
    diff_count = np.sum(sarsa_policy != ql_policy)
    print(f"\n✗ Policies DIFFER in {diff_count} states")

In [ ]:
# ============================================================================
# VISUALIZATION OF EXPERIMENT A
# ============================================================================
#
# TODO: run this cell after the previous one succeeds
# ============================================================================

# X axis: episode numbers (each point = average over WINDOW episodes)
episode_axis = np.arange(len(rewards_sarsa_std)) * WINDOW

# Create a figure with two plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ============================================================================
# Plot 1: Average reward per episode
# ============================================================================
axes[0].plot(episode_axis, rewards_sarsa_std, label='SARSA', linewidth=2, color='blue')
axes[0].plot(episode_axis, rewards_ql_std, label='Q-learning', linewidth=2, color='orange')
axes[0].set_title('Experiment A: Average reward', fontsize=14)
axes[0].set_xlabel('Episodes')
axes[0].set_ylabel(f'Average reward (window {WINDOW})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ============================================================================
# Plot 2: Success Rate (fraction of successful episodes)
# ============================================================================
axes[1].plot(episode_axis, np.array(success_sarsa_std) * 100,
             label='SARSA', linewidth=2, color='blue')
axes[1].plot(episode_axis, np.array(success_ql_std) * 100,
             label='Q-learning', linewidth=2, color='orange')
axes[1].set_title('Experiment A: Success Rate', fontsize=14)
axes[1].set_xlabel('Episodes')
axes[1].set_ylabel('Successful episodes, %')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 105)  # Limit the Y axis for clarity

plt.tight_layout()
plt.show()

# Print the final metrics
print("\n" + "=" * 50)
print("RESULTS OF EXPERIMENT A")
print("=" * 50)
print(f"SARSA      — final success rate: {success_sarsa_std[-1] * 100:.1f}%")
print(f"Q-learning — final success rate: {success_ql_std[-1] * 100:.1f}%")

### Conclusions for Experiment A
- TODO: Which algorithm achieved better final performance?
- TODO: Do you observe a difference in convergence speed?
- TODO: How do the learned policies differ? (check `np.argmax(Q_sarsa, axis=1)` vs `np.argmax(Q_ql, axis=1)`)

## 6. Experiment B: Effect of a negative reward for holes

Now let's use the modified environment with `hole_penalty=-1.0` and compare the algorithms' behavior.

### Why do we need a hole penalty?
In standard FrozenLake, the reward for falling into a hole = 0 (the same as for a regular step).
The agent doesn't "fear" holes — it simply doesn't get a positive reward.

By adding a **negative reward** for holes:
- The agent starts to **avoid** dangerous cells
- SARSA (on-policy) accounts for actual falls during exploration
- Q-learning (off-policy) only estimates optimal behavior

### Hypothesis
| Algorithm | Expected behavior |
|----------|---------------------|
| **SARSA** | More **conservative** — avoids risky paths, since it accounts for ε-random falls |
| **Q-learning** | More **aggressive** — chooses short routes, ignoring exploration risk |

### Experiment plan
1. Create an environment with `hole_penalty=-1.0`
2. Train both algorithms
3. Compare:
   - Average reward (will be lower due to penalties)
   - Success rate
   - Policies (SARSA should route around holes by a wider margin)

In [ ]:
# ============================================================================
# EXPERIMENT B: Effect of the hole-falling penalty
# ============================================================================
#
# TODO: fill in the blanks (...) and run the cell
# ============================================================================

print("=" * 50)
print("EXPERIMENT B: Environment with a hole penalty")
print("=" * 50)
print(f"hole_penalty = -1.0 (standard: 0.0)")
print()

# ============================================================================
# Step 1: Train SARSA on the modified environment
# ============================================================================
print("Training SARSA + penalty...")

# Create the environment with a hole penalty
# TODO: create the environment with hole_penalty=-1.0
env_penalty = make_env(hole_penalty=..., is_slippery=False)  # -1.0

# Run training (use the same configuration)
Q_sarsa_penalty, rewards_sarsa_penalty, success_sarsa_penalty = sarsa(env_penalty, sarsa_config)
env_penalty.close()

print(f"SARSA + penalty — final success rate: {success_sarsa_penalty[-1] * 100:.1f}%")

# ============================================================================
# Step 2: Train Q-learning on the modified environment
# ============================================================================
print("\nTraining Q-learning + penalty...")

# Create the same environment
env_penalty = make_env(hole_penalty=-1.0, is_slippery=False)

# Run training
Q_ql_penalty, rewards_ql_penalty, success_ql_penalty = q_learning(env_penalty, ql_config)
env_penalty.close()

print(f"Q-learning + penalty — final success rate: {success_ql_penalty[-1] * 100:.1f}%")

# ============================================================================
# Step 3: Compare with the standard environment
# ============================================================================
print("\n" + "=" * 50)
print("COMPARISON: standard environment vs penalty")
print("=" * 50)
print(f"{'Algorithm':<20} | {'Standard':>10} | {'+ Penalty':>10}")
print("-" * 45)
print(f"{'SARSA':<20} | {success_sarsa_std[-1]*100:>9.1f}% | {success_sarsa_penalty[-1]*100:>9.1f}%")
print(f"{'Q-learning':<20} | {success_ql_std[-1]*100:>9.1f}% | {success_ql_penalty[-1]*100:>9.1f}%")

In [ ]:
# ============================================================================
# VISUALIZATION OF EXPERIMENT B: Comparison of 4 variants (2x2 grid)
# ============================================================================
#
# Rows: standard environment / penalty environment
# Columns: average reward / success rate
# ============================================================================

episode_axis = np.arange(len(rewards_sarsa_std)) * WINDOW

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ============================================================================
# Row 1: Standard environment (Experiment A)
# ============================================================================
axes[0, 0].plot(episode_axis, rewards_sarsa_std, label='SARSA', linewidth=2)
axes[0, 0].plot(episode_axis, rewards_ql_std, label='Q-learning', linewidth=2)
axes[0, 0].set_title('Standard environment — Reward', fontsize=12)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(episode_axis, np.array(success_sarsa_std) * 100, label='SARSA', linewidth=2)
axes[0, 1].plot(episode_axis, np.array(success_ql_std) * 100, label='Q-learning', linewidth=2)
axes[0, 1].set_title('Standard environment — Success Rate', fontsize=12)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim(0, 105)

# ============================================================================
# Row 2: Environment with a penalty (Experiment B)
# ============================================================================
axes[1, 0].plot(episode_axis, rewards_sarsa_penalty, label='SARSA + penalty', linewidth=2)
axes[1, 0].plot(episode_axis, rewards_ql_penalty, label='Q-learning + penalty', linewidth=2)
axes[1, 0].set_title('Penalty environment — Reward', fontsize=12)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
# Note: the reward can be negative!

axes[1, 1].plot(episode_axis, np.array(success_sarsa_penalty) * 100, label='SARSA + penalty', linewidth=2)
axes[1, 1].plot(episode_axis, np.array(success_ql_penalty) * 100, label='Q-learning + penalty', linewidth=2)
axes[1, 1].set_title('Penalty environment — Success Rate', fontsize=12)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim(0, 105)

# Axis labels
for ax in axes.flat:
    ax.set_xlabel('Episodes')
for ax in axes[:, 0]:
    ax.set_ylabel('Average reward')
for ax in axes[:, 1]:
    ax.set_ylabel('Success Rate, %')

plt.suptitle('Experiment B: Effect of the hole penalty', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# POLICY VISUALIZATION (maps of optimal actions)
# ============================================================================
#
# Show which action the agent chooses in each state
# ============================================================================

def visualize_policy(Q: np.ndarray, title: str, ax=None) -> None:
    """Displays the optimal action for each cell.

    FrozenLake 4x4 map:
        S F F F      S = Start (state 0)
        F H F H      F = Frozen (ice)
        F F F H      H = Hole
        H F F G      G = Goal (state 15)
    """
    actions = ['←', '↓', '→', '↑']  # 0=Left, 1=Down, 2=Right, 3=Up

    # Extract the policy: for each state take the action with max Q
    policy = np.argmax(Q, axis=1).reshape(4, 4)

    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 4))

    # Draw arrows for each cell
    for i in range(4):
        for j in range(4):
            state = i * 4 + j
            ax.text(j, i, actions[policy[i, j]],
                   ha='center', va='center', fontsize=20, fontweight='bold')

    # Configure the axes
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xlim(-0.5, 3.5)
    ax.set_ylim(-0.5, 3.5)
    ax.grid(True, linewidth=2)
    ax.set_title(title, fontsize=11)
    ax.invert_yaxis()  # So that (0,0) is at the top left


# Create a 2x2 grid for all 4 policies
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

visualize_policy(Q_sarsa_std, 'SARSA — standard environment', axes[0, 0])
visualize_policy(Q_ql_std, 'Q-learning — standard environment', axes[0, 1])
visualize_policy(Q_sarsa_penalty, 'SARSA — penalty environment', axes[1, 0])
visualize_policy(Q_ql_penalty, 'Q-learning — penalty environment', axes[1, 1])

plt.suptitle('Policy comparison: SARSA vs Q-learning', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# ============================================================================
# Analysis of policy differences
# ============================================================================
print("\n" + "=" * 50)
print("POLICY ANALYSIS")
print("=" * 50)

# Compare policies pairwise
policies = {
    'SARSA std': np.argmax(Q_sarsa_std, axis=1),
    'Q-learning std': np.argmax(Q_ql_std, axis=1),
    'SARSA penalty': np.argmax(Q_sarsa_penalty, axis=1),
    'Q-learning penalty': np.argmax(Q_ql_penalty, axis=1)
}

print("\nDifferences between policies (number of states with different actions):")
names = list(policies.keys())
for i in range(len(names)):
    for j in range(i+1, len(names)):
        diff = np.sum(policies[names[i]] != policies[names[j]])
        print(f"  {names[i]} vs {names[j]}: {diff} differences")

### Conclusions for Experiment B
- TODO: How did SARSA's behavior change when the penalty was added?
- TODO: How did Q-learning's behavior change?
- TODO: Which algorithm showed safer behavior?
- TODO: Explain the difference in average rewards between the algorithms

## 7. Experiment C: Softmax vs ε-greedy policy

Let's investigate the effect of the **exploration policy type** on SARSA's performance.

### Two approaches to exploration

| Policy | Description | Formula |
|----------|----------|---------|
| **ε-greedy** | With probability ε — a random action, otherwise the best one | $P(a) = \begin{cases} 1-\varepsilon + \varepsilon/n & \text{if } a = \arg\max Q \\ \varepsilon/n & \text{otherwise} \end{cases}$ |
| **Softmax** | Probabilities proportional to exp(Q/τ) | $P(a) = \frac{e^{Q(s,a)/\tau}}{\sum_{a'} e^{Q(s,a')/\tau}}$ |

### Effect of temperature τ in Softmax

| Temperature | Behavior |
|-------------|-----------|
| τ → 0 | Nearly greedy choice (argmax) — little exploration |
| τ = 1.0 | Balanced behavior |
| τ → ∞ | Nearly uniform choice — lots of exploration |

### Experiment plan
1. Train SARSA with **ε-greedy** (decay: 1.0 → 0.01) — already done in Experiment A
2. Train SARSA with **Softmax** at different temperatures: τ = 0.5, 1.0, 2.0
3. Compare convergence speed and final success rate

### Hypothesis
- **τ = 0.5** (low): fast convergence, but risk of getting stuck in a local optimum
- **τ = 1.0** (medium): good exploration/exploitation balance
- **τ = 2.0** (high): a lot of randomness, slow convergence

In [ ]:
# ============================================================================
# EXPERIMENT C: Comparison of exploration policies (ε-greedy vs Softmax)
# ============================================================================
#
# TODO: fill in the blanks (...) and run the cell
# ============================================================================

from typing import Dict, List

print("=" * 50)
print("EXPERIMENT C: ε-greedy vs Softmax")
print("=" * 50)

# Temperatures to test
temperatures = [0.5, 1.0, 2.0]

# Dictionary to store results
softmax_results: Dict[float, Dict] = {}

# ============================================================================
# Training SARSA with different Softmax temperatures
# ============================================================================
for temp in temperatures:
    print(f"\n--- Training SARSA with Softmax (τ={temp}) ---")

    # Create the environment
    env = make_env(hole_penalty=0.0, is_slippery=False)

    # Train SARSA with the softmax policy
    # TODO: call sarsa() with use_softmax=True and temperature=temp
    Q_soft, rewards_soft, success_soft = sarsa(
        env,
        sarsa_config,
        use_softmax=...,      # TODO: True
        temperature=...       # TODO: temp
    )
    env.close()

    # Save the results
    softmax_results[temp] = {
        'Q': Q_soft,
        'rewards': rewards_soft,
        'success': success_soft
    }

    print(f"Softmax τ={temp}: final success rate = {success_soft[-1] * 100:.1f}%")

# ============================================================================
# Comparison with ε-greedy (already trained in Experiment A)
# ============================================================================
print("\n" + "=" * 50)
print("RESULTS OF EXPERIMENT C")
print("=" * 50)

print(f"\n{'Policy':<25} | {'Final Success Rate':>20}")
print("-" * 50)
print(f"{'ε-greedy (decay)':<25} | {success_sarsa_std[-1] * 100:>19.1f}%")

for temp in temperatures:
    sr = softmax_results[temp]['success'][-1] * 100
    print(f"{'Softmax τ=' + str(temp):<25} | {sr:>19.1f}%")

In [ ]:
# ============================================================================
# VISUALIZATION OF EXPERIMENT C: Comparison of exploration policies
# ============================================================================

episode_axis = np.arange(len(success_sarsa_std)) * WINDOW

plt.figure(figsize=(12, 6))

# ε-greedy (baseline from Experiment A)
plt.plot(episode_axis, np.array(success_sarsa_std) * 100,
         label='ε-greedy (decay 1.0→0.01)', linewidth=2.5, color='black', linestyle='--')

# Softmax with different temperatures
colors = ['red', 'green', 'blue']
for temp, color in zip(temperatures, colors):
    sr = np.array(softmax_results[temp]['success']) * 100
    plt.plot(episode_axis, sr, label=f'Softmax τ={temp}', linewidth=2, color=color)

plt.title('Experiment C: Effect of the exploration policy on SARSA training', fontsize=14)
plt.xlabel('Episodes')
plt.ylabel('Success Rate, %')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim(0, 105)

plt.tight_layout()
plt.show()

# ============================================================================
# Analysis: which policy is better?
# ============================================================================
print("\n" + "=" * 50)
print("RESULT ANALYSIS")
print("=" * 50)

# Find the best policy
all_results = {'ε-greedy': success_sarsa_std[-1]}
for temp in temperatures:
    all_results[f'Softmax τ={temp}'] = softmax_results[temp]['success'][-1]

best_policy = max(all_results, key=all_results.get)
print(f"\nBest policy: {best_policy} ({all_results[best_policy]*100:.1f}%)")

# Convergence speed (episode when success rate first exceeded 50%)
print("\nConvergence speed (episode when SR > 50%):")
for name, sr_list in [('ε-greedy', success_sarsa_std)] + \
                     [(f'Softmax τ={t}', softmax_results[t]['success']) for t in temperatures]:
    sr_array = np.array(sr_list)
    idx = np.where(sr_array > 0.5)[0]
    if len(idx) > 0:
        episode = idx[0] * WINDOW
        print(f"  {name:<20}: episode {episode}")
    else:
        print(f"  {name:<20}: did not reach 50%")

### Conclusions for Experiment C
- TODO: Which softmax temperature showed the best result?
- TODO: How does temperature affect the exploration/exploitation balance?
- TODO: In which cases is softmax preferable to ε-greedy?

**Expected observations:**
- Low temperature (τ=0.5): more greedy behavior, may get stuck in a local optimum
- Medium temperature (τ=1.0): good balance, close to ε-greedy
- High temperature (τ=2.0): too much exploration, slow convergence
- Softmax transitions more smoothly from exploration to exploitation

## 8. Additional analysis (optional)

**Tasks for further study:**

1. **Trajectory analysis**: record several episodes of the trained policies and visualize the routes
2. **Visitation matrix**: build a heatmap of state visitation frequency for different algorithms
3. **Sensitivity to α**: investigate the effect of the learning rate on convergence
4. **Double Q-learning**: implement and compare it with regular Q-learning
5. **Q-value analysis**: visualize the difference in Q(s,a) between SARSA and Q-learning

In [ ]:
# Additional analysis: evaluating greedy policies
#
# TODO (optional): uncomment for a final check

# def evaluate_greedy_policy(Q: np.ndarray, hole_penalty: float = 0.0, episodes: int = 200) -> Tuple[int, int]:
#     """Evaluates the greedy policy based on the Q-table.
#
#     Returns:
#         (hole_hits, goal_hits): number of falls into a hole and goal reaches
#     """
#     env = make_env(hole_penalty=hole_penalty)
#     hole_hits = 0
#     goal_hits = 0
#
#     for ep in range(episodes):
#         state, _ = env.reset(seed=SEED + 9000 + ep)
#         for _ in range(200):
#             action = int(np.argmax(Q[state]))  # Greedy action
#             state, reward, terminated, truncated, _ = env.step(action)
#             if terminated:
#                 if reward > 0:
#                     goal_hits += 1
#                 elif reward < 0:
#                     hole_hits += 1
#                 break
#             if truncated:
#                 break
#
#     env.close()
#     return hole_hits, goal_hits

# # Evaluate all 4 policies
# table = [
#     ('SARSA std', *evaluate_greedy_policy(Q_sarsa_std, 0.0)),
#     ('Q-learning std', *evaluate_greedy_policy(Q_ql_std, 0.0)),
#     ('SARSA penalty', *evaluate_greedy_policy(Q_sarsa_penalty, -1.0)),
#     ('Q-learning penalty', *evaluate_greedy_policy(Q_ql_penalty, -1.0)),
# ]

# print('Evaluation of greedy policies (200 episodes):')
# print(f"{'Algorithm':18s} | {'Goals':>5s} | {'Holes':>5s}")
# print('-' * 35)
# for name, holes, goals in table:
#     print(f"{name:18s} | {goals:5d} | {holes:5d}")

pass

## 9. Self-check questions

1. **TODO:** Why is SARSA called on-policy and Q-learning off-policy? Explain with an example.

2. **TODO:** In which situations is SARSA preferable to Q-learning? Give examples.

3. **TODO:** How does a negative reward for falling into a hole affect the algorithms' behavior?

4. **TODO:** Why can softmax with a low temperature lead to worse results?

5. **TODO:** What happens if you set α=1.0? Will the algorithm converge?